# CNN Model

## Preprocessing
This is a pipeline that is used for the CNN data to:
1. Extract and encode the DNA window around each CpG from CAST or 129 (depending on the particular mouse being used), and encode as a one-hot vector. This creates an input with shape `[examples, channels, window_length]`.
2. Find its corresponding target and methylated count and store it
3. Split data by genomic region rather than random rows to prevent similar nearby sites being in training and testing, making testing look better than it actually is.

We will start by predicting one allele's observed CpG methylation from mm9 windows (modified for the particular strain we are testing).

Each example is a CpG observation for one strain allele in one reciprocal cross. We combine the complementary Watson/Crick read counts into one CpG-level target; strand is not a model feature.

- **Input:** A 201 bp one-hot-encoded mm9 window from the matching strain consensus FASTA (note 129 FASTA available here is 129S1/SvImJ, a proxy for the actual 129X1/SvJ strain).
- **Target:** The observed methylation fraction for the matching strain allele in F1i or F1r. The input is sequence-only: cross and parent-of-origin are retained only for evaluating results, not fed to the CNN.
- **Coverage:** Methylated and unmethylated read counts are retained. The binary cross-entropy loss on each observed fraction is weighted by its coverage, corresponding to a binomial read-level likelihood.
- **Split:** Hold out chromosome 18 for validation and chromosome 19 for the final test; train on other autosomes. This prevents nearby CpGs and their overlapping windows from leaking between splits.

We use a deterministic genomic-coordinate sample and cap the number of CpG sites per split so the tutorial-sized run fits on an ordinary workstation. Adjust the settings in the first code cell for a larger run. Because samples are pooled, evaluation is descriptive and is not a substitute for biological replication.


In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from pyfaidx import Fasta

SEED = 23
torch.manual_seed(SEED)
WINDOW_LENGTH = 201
MIN_COVERAGE = 10
SAMPLE_PERCENT = 2
MAX_TRAIN_CPGS = 6_000
MAX_VALIDATION_CPGS = 1_500
MAX_TEST_CPGS = 1_500
BATCH_SIZE = 256
EPOCHS = 8
LEARNING_RATE = 1e-3
POSITIVE_THRESHOLD = 0.5

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "genetic_data").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the project directory or a subdirectory.")

ROOT = find_project_root()
METHYLATION_PATH = ROOT / "genetic_data/processed/methylation_cpg_counts_mm9.tsv.gz"

def first_existing(paths, description):
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Could not find {description}: " + ", ".join(map(str, paths)))

CAST_FASTA_PATH = first_existing(
    [ROOT / "genetic_data/processed/ref/mm9_CAST.fa",
     ROOT / "genetic_data/processed/ref/mm9_CAST_EiJ.fa",
     ROOT / "genetic_data/ref/mm9_CAST_EiJ.fa"],
    "CAST mm9 FASTA",
)
STRAIN129_FASTA_PATH = first_existing(
    [ROOT / "genetic_data/processed/ref/mm9_129.fa",
     ROOT / "genetic_data/processed/ref/mm9_129S1_SvImJ.fa",
     ROOT / "genetic_data/ref/mm9_129S1_SvImJ.fa"],
    "129 mm9 FASTA",
)
if not METHYLATION_PATH.is_file():
    raise FileNotFoundError(f"Methylation calls not found: {METHYLATION_PATH}")

print(f"Project: {ROOT}")
print(f"Methylation data: {METHYLATION_PATH}")
print(f"129 sequence: {STRAIN129_FASTA_PATH}")
print(f"CAST sequence: {CAST_FASTA_PATH}")
print(f"PyTorch: {torch.__version__}; device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Read only the call counts needed to form targets. The processed file has one
# row per cross, strand, and CpG observation; combine Watson and Crick below.
AUTOSOMES = {f"chr{i}" for i in range(1, 20)}
COUNT_COLUMNS = ["129_methylated", "129_unmethylated", "Cast_methylated", "Cast_unmethylated"]
USE_COLUMNS = ["cross", "strand", "chrom", "pos_as_reported", "context", *COUNT_COLUMNS]
partials = []
rows_read = 0
for chunk in pd.read_csv(
    METHYLATION_PATH,
    sep="\t",
    compression="gzip",
    usecols=USE_COLUMNS,
    chunksize=500_000,
    dtype={"pos_as_reported": "int64", **{column: "int32" for column in COUNT_COLUMNS}},
):
    rows_read += len(chunk)
    keep = chunk["chrom"].isin(AUTOSOMES) & chunk["context"].eq("CG")
    if not keep.any():
        continue
    part = chunk.loc[keep, ["cross", "strand", "chrom", "pos_as_reported", *COUNT_COLUMNS]].copy()
    # Coordinates are 1-based: Watson is on the CpG C; Crick is on its G.
    # Thus crick_pos - 1 and watson_pos identify the same CpG C coordinate.
    part["cpg"] = part["pos_as_reported"] - part["strand"].eq("crick").astype("int64")
    part["chrom"] = part["chrom"].astype(str)
    chrom_number = part["chrom"].str[3:].to_numpy(dtype=np.uint64)
    coordinate_hash = (
        part["cpg"].to_numpy(dtype=np.uint64) * np.uint64(2_654_435_761)
        + chrom_number * np.uint64(2_246_822_519)
    ) % np.uint64(100)
    part = part.loc[coordinate_hash < SAMPLE_PERCENT]
    if part.empty:
        continue
    partials.append(
        part.groupby(["cross", "chrom", "cpg"], sort=False, observed=True)[COUNT_COLUMNS]
        .sum()
        .reset_index()
    )
    del chunk, part
    gc.collect()

if not partials:
    raise ValueError("No autosomal CpG calls were selected; check the methylation file and column names.")
site_cross = (
    pd.concat(partials, ignore_index=True)
    .groupby(["cross", "chrom", "cpg"], sort=False, observed=True)[COUNT_COLUMNS]
    .sum()
    .reset_index()
)
del partials

# Require adequate coverage for both strain alleles in each cross, then keep
# only CpGs that pass this filter in both reciprocal crosses.
site_cross["coverage_129"] = site_cross["129_methylated"] + site_cross["129_unmethylated"]
site_cross["coverage_CAST"] = site_cross["Cast_methylated"] + site_cross["Cast_unmethylated"]
eligible = site_cross.loc[
    (site_cross["coverage_129"] >= MIN_COVERAGE)
    & (site_cross["coverage_CAST"] >= MIN_COVERAGE)
].copy()
paired_sites = (
    eligible.groupby(["chrom", "cpg"], observed=True)["cross"]
    .nunique()
    .rename("n_crosses")
    .reset_index()
    .query("n_crosses == 2")
    [["chrom", "cpg"]]
)
eligible = eligible.merge(paired_sites, on=["chrom", "cpg"], validate="many_to_one")
eligible["split"] = np.select(
    [eligible["chrom"].eq("chr18"), eligible["chrom"].eq("chr19")],
    ["validation", "test"],
    default="train",
)
# Select CpG coordinates, not individual rows, and keep both crosses together.
site_keys = eligible[["chrom", "cpg", "split"]].drop_duplicates()
site_limits = {"train": MAX_TRAIN_CPGS, "validation": MAX_VALIDATION_CPGS, "test": MAX_TEST_CPGS}
selected_keys = []
for split, limit in site_limits.items():
    candidates = site_keys.loc[site_keys["split"].eq(split)]
    selected_keys.append(candidates.sample(n=min(limit, len(candidates)), random_state=SEED))
selected_keys = pd.concat(selected_keys, ignore_index=True)
examples = eligible.merge(selected_keys, on=["chrom", "cpg", "split"], validate="many_to_one")
examples["129_coverage"] = examples["coverage_129"]
examples["CAST_coverage"] = examples["coverage_CAST"]
print(f"Scanned {rows_read:,} rows; sampled {len(site_cross):,} CpG-by-cross count records.")
print(f"CpGs with both alleles adequately covered in both crosses: {len(paired_sites):,}")
print(f"Selected {len(selected_keys):,} genomic CpGs -> {len(examples) * 2:,} strain/cross examples before sequence QC.")
print(selected_keys["split"].value_counts().to_string())


## Model

Create a basic CNN with Conv1D layers, ReLU, global max pooling, and a sigmoid output for the methylation fraction.

Weight the loss by coverage.

In [ ]:
class MethylationCNN(torch.nn.Module):
    """Small 1D CNN mapping a one-hot DNA window to a methylation fraction."""
    def __init__(self, input_channels=5):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv1d(input_channels, 32, kernel_size=7, padding=3),
            torch.nn.ReLU(),
            torch.nn.Conv1d(32, 64, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.AdaptiveMaxPool1d(1),
            torch.nn.Flatten(),
        )
        self.head = torch.nn.Sequential(
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
            torch.nn.Sigmoid(),
        )

    def forward(self, sequence):
        return self.head(self.features(sequence)).squeeze(-1)

print(MethylationCNN())


## Encode strain windows and create data loaders

For every selected CpG and cross, create one example from the 129 sequence and one from the CAST sequence. Their observed read counts provide the labels. Windows with an `N`, an incomplete edge window, or no `CG` at the center in that allele's FASTA are excluded because they do not represent a valid CpG for that sequence. Training/validation/test assignment is inherited from the whole chromosome.

In [ ]:
# Open the large genome FASTAs only after the methylation loci have been sampled.
# pyfaidx creates/reuses an adjacent .fai index for random access.
fasta_129 = Fasta(str(STRAIN129_FASTA_PATH), as_raw=True, sequence_always_upper=True)
fasta_CAST = Fasta(str(CAST_FASTA_PATH), as_raw=True, sequence_always_upper=True)

BASE_CODE = np.full(256, 4, dtype=np.uint8)  # 4 is the explicit unknown/N channel.
for base, code in [("A", 0), ("C", 1), ("G", 2), ("T", 3)]:
    BASE_CODE[ord(base)] = code
    BASE_CODE[ord(base.lower())] = code
ONE_HOT = np.eye(5, dtype=np.float32)
half = WINDOW_LENGTH // 2
window_cache = {}
records = {name: [] for name in ["train", "validation", "test"]}
sequence_qc = {"missing_chromosome": 0, "short_window": 0, "no_cpg_motif": 0}

def sequence_codes(fasta, strain, chrom, cpg):
    key = (strain, chrom, int(cpg))
    if key not in window_cache:
        center0 = int(cpg) - 1  # convert the 1-based CpG C coordinate to 0-based indexing
        start, end = center0 - half, center0 + half + 1
        if start < 0:
            sequence_qc["short_window"] += 1
            return None
        try:
            text = str(fasta[chrom][start:end]).upper()
        except KeyError:
            sequence_qc["missing_chromosome"] += 1
            return None
        if len(text) != WINDOW_LENGTH or "N" in text:
            sequence_qc["short_window"] += 1
            return None
        # A reported CpG needs to be a CpG on the corresponding haplotype.
        if text[half:half + 2] != "CG":
            sequence_qc["no_cpg_motif"] += 1
            return None
        window_cache[key] = BASE_CODE[np.frombuffer(text.encode("ascii"), dtype=np.uint8)]
    return window_cache[key]

for row in examples.to_dict("records"):
    for strain, fasta, methylated_column, unmethylated_column in [
        ("129", fasta_129, "129_methylated", "129_unmethylated"),
        ("CAST", fasta_CAST, "Cast_methylated", "Cast_unmethylated"),
    ]:
        codes = sequence_codes(fasta, strain, row["chrom"], row["cpg"])
        if codes is None:
            continue
        methylated = int(row[methylated_column])
        unmethylated = int(row[unmethylated_column])
        coverage = methylated + unmethylated
        records[row["split"]].append(
            (codes, methylated / coverage, coverage, row["chrom"], row["cpg"], row["cross"], strain)
        )

print("Sequence filtering:", sequence_qc)
print("Cached distinct strain windows:", f"{len(window_cache):,}")

def make_dataset(rows):
    if not rows:
        raise ValueError("A data split is empty after sequence QC; check reference paths and selected chromosomes.")
    codes = np.stack([record[0] for record in rows])
    x = ONE_HOT[codes].transpose(0, 2, 1).copy()  # [examples, channels, bases]
    y = np.asarray([record[1] for record in rows], dtype=np.float32)
    coverage = np.asarray([record[2] for record in rows], dtype=np.float32)
    metadata = pd.DataFrame(
        [(r[3], r[4], r[5], r[6]) for r in rows],
        columns=["chrom", "cpg", "cross", "strain"],
    )
    return TensorDataset(torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(coverage)), metadata

datasets = {}
metadata = {}
for split in ["train", "validation", "test"]:
    datasets[split], metadata[split] = make_dataset(records[split])
    print(f"{split:10s}: {len(datasets[split]):,} examples, X={tuple(datasets[split].tensors[0].shape)}")

train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MethylationCNN(input_channels=ONE_HOT.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
print(f"Training on {device}; sequence-only inputs include strain-specific bases but no parent/cross label.")


## Training

Train with coverage-weighted binary cross-entropy on the observed methylation fraction. This estimates a probability/fraction in `[0, 1]`; it is a regression-style probability task, **not** ordinary binary classification. For the requested F1/accuracy/precision/recall, we additionally threshold observed and predicted fractions at 0.5 and report those as secondary diagnostics. The main validation/test metrics are weighted BCE, MAE, and RMSE. The checkpoint with lowest validation loss is evaluated once on the held-out chromosome 19.

In [ ]:
def weighted_bce(prediction, target, coverage):
    per_example = torch.nn.functional.binary_cross_entropy(prediction, target, reduction="none")
    return torch.sum(per_example * coverage) / torch.sum(coverage)

@torch.no_grad()
def evaluate(model, loader, threshold=POSITIVE_THRESHOLD):
    model.eval()
    predictions, targets, coverages = [], [], []
    loss_numerator = 0.0
    total_coverage = 0.0
    for x, y, coverage in loader:
        x, y, coverage = x.to(device), y.to(device), coverage.to(device)
        prediction = model(x)
        loss_numerator += torch.nn.functional.binary_cross_entropy(
            prediction, y, reduction="none"
        ).mul(coverage).sum().item()
        total_coverage += coverage.sum().item()
        predictions.append(prediction.cpu().numpy())
        targets.append(y.cpu().numpy())
        coverages.append(coverage.cpu().numpy())
    prediction = np.concatenate(predictions)
    target = np.concatenate(targets)
    coverage = np.concatenate(coverages)
    predicted_positive = prediction >= threshold
    actual_positive = target >= threshold
    tp = int(np.sum(predicted_positive & actual_positive))
    fp = int(np.sum(predicted_positive & ~actual_positive))
    fn = int(np.sum(~predicted_positive & actual_positive))
    tn = int(np.sum(~predicted_positive & ~actual_positive))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "weighted_bce": loss_numerator / total_coverage,
        "mae": float(np.average(np.abs(prediction - target), weights=coverage)),
        "rmse": float(np.sqrt(np.average((prediction - target) ** 2, weights=coverage))),
        "accuracy_at_0.5": (tp + tn) / len(target),
        "precision_at_0.5": precision,
        "recall_at_0.5": recall,
        "f1_at_0.5": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "examples": len(target),
    }

torch.manual_seed(SEED)
best_validation_loss = float("inf")
best_state = None
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_numerator = 0.0
    train_coverage_total = 0.0
    for x, y, coverage in train_loader:
        x, y, coverage = x.to(device), y.to(device), coverage.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = weighted_bce(model(x), y, coverage)
        loss.backward()
        optimizer.step()
        train_loss_numerator += loss.item() * coverage.sum().item()
        train_coverage_total += coverage.sum().item()
    validation_metrics = evaluate(model, validation_loader)
    train_loss = train_loss_numerator / train_coverage_total
    history.append({"epoch": epoch, "train_weighted_bce": train_loss, **validation_metrics})
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | train weighted BCE {train_loss:.4f} | "
        f"val weighted BCE {validation_metrics['weighted_bce']:.4f} | "
        f"MAE {validation_metrics['mae']:.4f} | RMSE {validation_metrics['rmse']:.4f} | "
        f"F1 {validation_metrics['f1_at_0.5']:.3f} | "
        f"accuracy {validation_metrics['accuracy_at_0.5']:.3f} | "
        f"precision {validation_metrics['precision_at_0.5']:.3f} | "
        f"recall {validation_metrics['recall_at_0.5']:.3f}"
    )
    if validation_metrics["weighted_bce"] < best_validation_loss:
        best_validation_loss = validation_metrics["weighted_bce"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

if best_state is None:
    raise RuntimeError("Training did not produce a validation checkpoint.")
model.load_state_dict(best_state)
test_metrics = evaluate(model, test_loader)
print("\nHeld-out chr19 test metrics (best validation checkpoint):")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

# A coverage-weighted training-mean prediction is a simple reference point.
train_targets = datasets["train"].tensors[1].numpy()
train_coverage = datasets["train"].tensors[2].numpy()
test_targets = datasets["test"].tensors[1].numpy()
test_coverage = datasets["test"].tensors[2].numpy()
baseline_prediction = float(np.average(train_targets, weights=train_coverage))
baseline_clipped = np.clip(baseline_prediction, 1e-7, 1 - 1e-7)
baseline_metrics = {
    "prediction": baseline_prediction,
    "weighted_bce": float(np.average(
        -(test_targets * np.log(baseline_clipped) + (1 - test_targets) * np.log(1 - baseline_clipped)),
        weights=test_coverage,
    )),
    "mae": float(np.average(np.abs(test_targets - baseline_prediction), weights=test_coverage)),
    "rmse": float(np.sqrt(np.average((test_targets - baseline_prediction) ** 2, weights=test_coverage))),
}
print("\nHeld-out chr19 baseline (predict the coverage-weighted training mean):")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

OUTPUT_PATH = ROOT / "models/cnn/cnn_methylation_model.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "window_length": WINDOW_LENGTH,
        "input_channels": int(ONE_HOT.shape[1]),
        "threshold": POSITIVE_THRESHOLD,
        "test_metrics": test_metrics,
        "baseline_metrics": baseline_metrics,
    },
    OUTPUT_PATH,
)
print(f"Saved model checkpoint to {OUTPUT_PATH}")
